# 06 — Cartes pour la démo

**Rôle dans le pipeline :** dessiner les résultats. Aucun calcul ici : on lit seulement les fichiers Silver.

| Lit | Produit (`resultats/cartes/`) |
|---|---|
| `silver/offre_arrets.parquet` | `carte_arrets.html` |
| `silver/carreaux_acces.parquet` | `carte_acces.html`, `carte_isoles.html` |
| `silver/emplacements.parquet` + `silver/selection.parquet` | `carte_selection.html` |

Les cartes sont **enregistrées en HTML** (à ouvrir dans le navigateur) plutôt qu'affichées dans le notebook : chaque carte contient des milliers de formes, et les garder dans le notebook l'alourdit.

Chaque carte porte sa **légende** (en bas à gauche), également affichée sous la cellule qui la produit. Les deux viennent de la même fonction : elles ne peuvent pas diverger.

In [1]:
import folium
import pandas as pd
from IPython.display import HTML, display
from pyproj import Transformer
from parametres import (SILVER, CARTES_DIR, COMMUNES, CRS_CARREAUX, TAILLE_CARREAU_M,
                        SEUIL_ISOLE_M, RAYON_STATION_M, RAYON_5MIN_M, FOND_CARTE,
                        SEUILS_OFFRE, NIVEAUX_OFFRE)

CARTES_DIR.mkdir(parents=True, exist_ok=True)
arrets = pd.read_parquet(SILVER / "offre_arrets.parquet")
carreaux = pd.read_parquet(SILVER / "emplacements.parquet")
selection = pd.read_parquet(SILVER / "selection.parquet")

CENTRE = [carreaux["lat"].mean(), carreaux["lon"].mean()]
COULEURS_NIVEAU = {
    "Aucun arrêt": "#4b0000",
    "Très faible": "#d7191c",
    "Faible": "#fdae61",
    "Correcte": "#a6d96a",
    "Forte": "#1a9641",
}


def enregistrer(carte, nom):
    carte.save(CARTES_DIR / nom)
    print("Écrit :", CARTES_DIR / nom)

C:\Users\nicol\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


### Légendes

`legende_html` construit la légende en HTML à partir d'une liste `(couleur, texte, forme)`. `ajouter_legende` la pose **sur la carte** (position fixe, en bas à gauche) et l'**affiche sous la cellule**.

Les libellés des niveaux d'offre sont **calculés à partir de `SEUILS_OFFRE`** : si on change un seuil dans `parametres.py`, la légende suit automatiquement.

In [2]:
FORMES = {
    "carre": "width:14px;height:14px;background:{c}",
    "rond": "width:10px;height:10px;border-radius:50%;background:{c}",
    "cercle": "width:12px;height:12px;border-radius:50%;border:2px solid {c}",
}


def legende_html(titre, elements, flottante=False):
    lignes = "".join(
        f'<div style="display:flex;align-items:center;gap:6px;margin:3px 0">'
        f'<span style="display:inline-block;flex:none;{FORMES[forme].format(c=couleur)}"></span>{texte}</div>'
        for couleur, texte, forme in elements
    )
    position = "position:fixed;bottom:30px;left:30px;z-index:9999;" if flottante else "display:inline-block;"
    return (f'<div style="{position}background:white;color:#222;padding:8px 10px;border:1px solid #999;'
            f'border-radius:4px;font:12px sans-serif">'
            f'<b>{titre}</b>{lignes}</div>')


def ajouter_legende(carte, titre, elements):
    carte.get_root().html.add_child(folium.Element(legende_html(titre, elements, flottante=True)))
    display(HTML(legende_html(titre, elements)))


def libelle_niveau(i):
    bas, haut = SEUILS_OFFRE[i] + 1, SEUILS_OFFRE[i + 1]
    if haut == 0:
        return f"{NIVEAUX_OFFRE[i]} (0 passage)"
    if haut == float("inf"):
        return f"{NIVEAUX_OFFRE[i]} ({bas} passages et plus)"
    return f"{NIVEAUX_OFFRE[i]} ({bas} à {haut:.0f} passages)"


TITRE_NIVEAUX = f"Offre de bus à moins de {RAYON_5MIN_M} m<br>(passages par jour, tous réseaux)"
ELEMENTS_NIVEAUX = [(COULEURS_NIVEAU[n], libelle_niveau(i), "carre") for i, n in enumerate(NIVEAUX_OFFRE)]

### Carte des arrêts

Couleur selon les passages du jour de référence. Les seuils sont les **quartiles mesurés** sur les données (25 % les moins desservis en rouge, 25 % les mieux desservis en vert), et non des valeurs fixées à la main.

In [3]:
q25, q75 = arrets["nb_passages"].quantile([0.25, 0.75])
carte = folium.Map(location=CENTRE, zoom_start=11, tiles=FOND_CARTE)
for a in arrets.itertuples():
    folium.CircleMarker(
        [a.stop_lat, a.stop_lon],
        radius=3,
        color="red" if a.nb_passages <= q25 else "green" if a.nb_passages > q75 else "orange",
        fill=True,
        fill_opacity=0.8,
        popup=f"{a.stop_name} ({a.reseau}) : {a.nb_passages} passages, {a.nb_lignes} lignes",
    ).add_to(carte)
ajouter_legende(carte, "Passages de bus par arrêt et par jour", [
    ("red", f"Peu desservi : {q25:.0f} ou moins (25 % des arrêts)", "rond"),
    ("orange", f"Moyen : {int(q25) + 1} à {int(q75)}", "rond"),
    ("green", f"Bien desservi : plus de {q75:.0f} (25 % des arrêts)", "rond"),
])
enregistrer(carte, "carte_arrets.html")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\resultats\cartes\carte_arrets.html


### Carte de l'accessibilité (objectif 1)

Chaque carreau de 200 m est dessiné à sa vraie taille. Ses coins sont calculés en mètres (centre ± 100 m), puis convertis en latitude/longitude.

In [4]:
transfo = Transformer.from_crs(CRS_CARREAUX, "EPSG:4326", always_xy=True)
demi = TAILLE_CARREAU_M / 2
lon_min, lat_min = transfo.transform(carreaux["x_m"].values - demi, carreaux["y_m"].values - demi)
lon_max, lat_max = transfo.transform(carreaux["x_m"].values + demi, carreaux["y_m"].values + demi)


def ajouter_carreaux(carte, opacite):
    for i, c in enumerate(carreaux.itertuples()):
        folium.Rectangle(
            bounds=[[lat_min[i], lon_min[i]], [lat_max[i], lon_max[i]]],
            stroke=False,
            fill=True,
            fill_color=COULEURS_NIVEAU[c.niveau_offre],
            fill_opacity=opacite,
            popup=(f"{c.ind:.0f} habitants | offre : {c.niveau_offre} | {c.nb_arrets_5min} arrêts à {RAYON_5MIN_M} m | "
                   f"{c.passages_5min} passages | arrêt le plus proche : {c.dist_arret_m:.0f} m"),
        ).add_to(carte)


carte = folium.Map(location=CENTRE, zoom_start=11, tiles=FOND_CARTE)
ajouter_carreaux(carte, 0.7)
ajouter_legende(carte, TITRE_NIVEAUX, ELEMENTS_NIVEAUX)
enregistrer(carte, "carte_acces.html")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\resultats\cartes\carte_acces.html


In [5]:
isoles = carreaux[carreaux["dist_arret_m"] > SEUIL_ISOLE_M]
carte = folium.Map(location=CENTRE, zoom_start=11, tiles=FOND_CARTE)
for c in isoles.itertuples():
    folium.CircleMarker(
        [c.lat, c.lon],
        radius=5,
        color="purple",
        fill=True,
        popup=f"{COMMUNES.get(c.commune, c.commune)} | {c.ind:.0f} habitants | {c.dist_arret_m:.0f} m de l'arrêt le plus proche",
    ).add_to(carte)
ajouter_legende(carte, "Carreaux isolés", [
    ("purple", f"Carreau habité à plus de {SEUIL_ISOLE_M} m de tout arrêt", "rond"),
])
enregistrer(carte, "carte_isoles.html")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\resultats\cartes\carte_isoles.html


### Carte des stations recommandées

Fond : les carreaux colorés par offre de bus (atténués). Par-dessus : chaque station retenue, son **rang** et son cercle de **500 m** (zone servie).

In [6]:
carte = folium.Map(location=CENTRE, zoom_start=11, tiles=FOND_CARTE)
ajouter_carreaux(carte, 0.35)
for s in selection.itertuples():
    folium.Circle([s.lat, s.lon], radius=RAYON_STATION_M, color="#1f4e9c", weight=2, fill=False).add_to(carte)
    folium.Marker(
        [s.lat, s.lon],
        icon=folium.DivIcon(html=f'<div style="background:#1f4e9c;color:white;border-radius:50%;width:22px;height:22px;'
                                 f'text-align:center;font:bold 12px/22px sans-serif">{s.rang}</div>'),
        popup=(f"#{s.rang} — {s.commune} | score {s.score_100:.0f}/100 | "
               f"{s.habitants_nouveaux:.0f} habitants nouvellement couverts | offre actuelle : {s.niveau_offre}"),
    ).add_to(carte)
ajouter_legende(carte, "Stations Citiz recommandées", [
    ("#1f4e9c", "Station (numéro = rang de priorité)", "rond"),
    ("#1f4e9c", f"Zone servie : {RAYON_STATION_M} m à pied", "cercle"),
] + ELEMENTS_NIVEAUX)
enregistrer(carte, "carte_selection.html")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\resultats\cartes\carte_selection.html
